# The Semantic Gap in Behavioral Embeddings
## Complete Reproducibility Pipeline

**Paper:** The Semantic Gap in Behavioral Embeddings: Why Linear Methods Fail for Educational RAG in Mathematics  
**Conference:** EDM 2026  
**Authors:** Ricky Gole, Jamell Dacon — Morgan State University

### Before running
1. Set runtime to GPU: Runtime > Change runtime type > T4 GPU
2. Mount your Google Drive
3. Set your OpenAI API key in Stage 0B
4. Confirm your data files are in the correct Drive locations (see README)

### Estimated time
- Stages 1-6: 30-45 minutes
- Stage 7 (DPO pairs): 20-30 minutes, ~$0.20
- Stage 8 (evaluation): 2-3 hours, ~$3-5
- Stages 9-11: 5 minutes

## Stage 0: Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'faiss-cpu',
    'sentence-transformers',
    'openai',
    'umap-learn',
    'transformers',
    'peft',
    'accelerate',
    'bitsandbytes',
    'trl',
    'datasets',
    'huggingface_hub'
])
print('Dependencies installed')

## Stage 0B: Imports and Setup

In [ ]:
import os
import json
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from scipy.sparse import csr_matrix
from scipy.stats import pearsonr, spearmanr, ttest_rel
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.manifold import TSNE
from google.colab import drive
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# Set your OpenAI API key here
OPENAI_KEY = ''  # <-- paste your key here, do not commit this to GitHub

BASE   = Path('/content/drive/MyDrive/edm')
OUT    = BASE / 'edm_outputs'
DATA   = BASE / 'data/train_data/train_task_1_2.csv'
KAGGLE = BASE / 'eedi-mining-misconceptions-in-mathematics/train.csv'
MISCMAP = BASE / 'eedi-mining-misconceptions-in-mathematics/misconception_mapping.csv'
OUT.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENAI_KEY)

print('Setup complete')
print(f'Base path: {BASE}')
print(f'OpenAI key set: {bool(OPENAI_KEY)}')

# Verify required data files
print()
for path, label in [
    (DATA,   'NeurIPS interaction data'),
    (KAGGLE, 'Kaggle question text'),
    (MISCMAP,'Misconception mapping'),
]:
    status = 'FOUND' if Path(path).exists() else 'MISSING - see README for download instructions'
    print(f'  {status}: {label}')

## Stage 1: Build Behavioral Manifold (SVD)

Constructs a 50-dimensional SVD embedding of the student-question interaction matrix. Skip if `behavioral_manifold.pkl` already exists in your Drive.

In [ ]:
manifold_path = OUT / 'behavioral_manifold.pkl'

if manifold_path.exists():
    print('behavioral_manifold.pkl found - loading')
    with open(manifold_path, 'rb') as f:
        manifold_data = pickle.load(f)
    embeddings   = manifold_data['embeddings']
    question_ids = manifold_data['question_ids']
    print(f'Loaded: {embeddings.shape}')
else:
    print('Building manifold from scratch (takes 10-15 minutes)...')
    df = pd.read_csv(
        DATA,
        dtype={'QuestionId': 'int32', 'UserId': 'int32', 'IsCorrect': 'int8'},
        usecols=['UserId', 'QuestionId', 'IsCorrect']
    )
    print(f'Interactions loaded: {len(df):,}')

    users     = df['UserId'].unique()
    questions = df['QuestionId'].unique()
    u2i = {u: i for i, u in enumerate(users)}
    q2i = {q: i for i, q in enumerate(questions)}
    df['ui'] = df['UserId'].map(u2i)
    df['qi'] = df['QuestionId'].map(q2i)

    M = csr_matrix(
        (df['IsCorrect'].values, (df['ui'].values, df['qi'].values)),
        shape=(len(users), len(questions)),
        dtype=np.float32
    )

    print('Running SVD (50 components)...')
    svd = TruncatedSVD(n_components=50, random_state=42)
    Q   = svd.fit_transform(M.T)
    Q   = normalize(Q, norm='l2')
    print(f'Explained variance: {svd.explained_variance_ratio_.sum():.2%}')

    manifold_data = {
        'embeddings':               Q,
        'question_ids':             questions,
        'question_to_idx':          q2i,
        'n_components':             50,
        'explained_variance_ratio': svd.explained_variance_ratio_,
        'singular_values':          svd.singular_values_
    }
    with open(manifold_path, 'wb') as f:
        pickle.dump(manifold_data, f)
    embeddings   = Q
    question_ids = questions
    print(f'Manifold saved: {embeddings.shape}')

## Stage 2: Spectral Clustering

Applies spectral clustering with nearest-neighbors affinity (n=10) to identify k=50 behavioral clusters, including high-density trap clusters. Reports real silhouette score.

In [ ]:
print('Running spectral clustering (nearest-neighbors affinity, k=50)...')
print('Takes 5-10 minutes on GPU...')

clustering = SpectralClustering(
    n_clusters=50,
    affinity='nearest_neighbors',
    n_neighbors=10,
    random_state=42,
    n_jobs=-1
)
cluster_labels = clustering.fit_predict(embeddings)

silhouette = silhouette_score(
    embeddings, cluster_labels,
    sample_size=5000, random_state=42
)
counts = Counter(cluster_labels)
print(f'Clustering complete')
print(f'Silhouette score: {silhouette:.4f}')
print(f'Min cluster size: {min(counts.values())}   Max: {max(counts.values())}   Median: {np.median(list(counts.values())):.0f}')

## Stage 3: Cluster Statistics and Trap Identification

Computes failure rates and interaction density for each cluster. Identifies trap clusters using a 40% failure rate and density threshold of 1000 interactions per question.

In [ ]:
print('Computing cluster statistics...')

df_int = pd.read_csv(
    DATA,
    dtype={'QuestionId': 'int32', 'UserId': 'int32', 'IsCorrect': 'int8'},
    usecols=['UserId', 'QuestionId', 'IsCorrect']
)

stats = []
for cid in range(50):
    mask  = cluster_labels == cid
    qids  = question_ids[mask]
    cdata = df_int[df_int['QuestionId'].isin(qids)]
    n_q   = len(qids)
    n_int = len(cdata)
    succ  = cdata['IsCorrect'].mean() if n_int > 0 else 0
    stats.append({
        'cluster_id':     cid,
        'n_questions':    n_q,
        'n_interactions': n_int,
        'n_students':     cdata['UserId'].nunique() if n_int > 0 else 0,
        'success_rate':   succ,
        'failure_rate':   1 - succ,
        'density':        n_int / n_q if n_q > 0 else 0
    })

cluster_df = pd.DataFrame(stats)

traps = cluster_df[
    (cluster_df['density']      > 1000) &
    (cluster_df['failure_rate'] > 0.40)
].sort_values('density', ascending=False).copy()

mastery = cluster_df[
    (cluster_df['success_rate'] > 0.70) &
    (cluster_df['density']      > 800)
].sort_values('success_rate', ascending=False)

print(f'Trap clusters identified: {len(traps)}')
print(traps[['cluster_id', 'failure_rate', 'density', 'n_questions']].to_string())
print(f'\nFailure rate range: {traps["failure_rate"].min():.1%} to {traps["failure_rate"].max():.1%}')
print(f'Mastery clusters (>70% success): {len(mastery)}')

## Stage 4: Label Clusters with Misconception Data

Cross-references behavioral clusters with expert misconception labels from the Eedi Kaggle dataset.

In [ ]:
print('Loading Kaggle data for misconception labeling...')
train_df = pd.read_csv(KAGGLE)
misc_df  = pd.read_csv(MISCMAP)

q2cluster = dict(zip(question_ids, cluster_labels))

mc_records = []
for _, row in train_df.iterrows():
    qid     = row['QuestionId']
    correct = row['CorrectAnswer']
    for ans in ['A', 'B', 'C', 'D']:
        if ans != correct:
            mc_id = row.get(f'Misconception{ans}Id')
            if pd.notna(mc_id):
                mc_records.append({'QuestionId': qid, 'MisconceptionId': int(mc_id)})

mc_map = pd.DataFrame(mc_records)
mc_map['cluster_id'] = mc_map['QuestionId'].map(q2cluster)
mc_map = mc_map.dropna(subset=['cluster_id'])
mc_map['cluster_id'] = mc_map['cluster_id'].astype(int)

cluster_misc = defaultdict(lambda: defaultdict(int))
for _, row in mc_map.iterrows():
    cluster_misc[row['cluster_id']][row['MisconceptionId']] += 1

label_rows = []
for cid in range(50):
    mcs = cluster_misc[cid]
    if not mcs:
        label_rows.append({'cluster_id': cid, 'dominant_misconception': None,
                           'dominant_percentage': 0.0, 'n_misconceptions': 0})
        continue
    total  = sum(mcs.values())
    dom_mc = max(mcs.items(), key=lambda x: x[1])
    label_rows.append({
        'cluster_id':             cid,
        'dominant_misconception': dom_mc[0],
        'dominant_count':         dom_mc[1],
        'dominant_percentage':    dom_mc[1] / total * 100,
        'n_misconceptions':       len(mcs)
    })

labels_df             = pd.DataFrame(label_rows)
cluster_stats_labeled = cluster_df.merge(labels_df, on='cluster_id', how='left')
trap_labeled          = cluster_stats_labeled[
    cluster_stats_labeled['cluster_id'].isin(traps['cluster_id'])
].copy()

labeled_results = {
    'cluster_labels':        cluster_labels,
    'cluster_stats_labeled': cluster_stats_labeled,
    'trap_clusters_labeled': trap_labeled,
    'cluster_misconceptions': dict(cluster_misc),
    'question_ids':           question_ids,
    'n_clusters':             50,
    'silhouette_score':       silhouette
}
with open(OUT / 'clustering_labeled_results.pkl', 'wb') as f:
    pickle.dump(labeled_results, f)
cluster_stats_labeled.to_csv(OUT / 'cluster_statistics_labeled.csv', index=False)

print(f'Clustering saved. Trap clusters labeled: {len(trap_labeled)}')

## Stage 5: Train/Eval Split

Selects 242 questions meeting high diagnostic utility criteria (500+ interactions, 40%+ failure rate, present in both NeurIPS and Kaggle datasets). Splits into 120 training and 122 evaluation questions.

In [ ]:
print('Computing per-question failure rates...')

kaggle_qids   = set(train_df['QuestionId'].tolist())
manifold_qids = set(question_ids)

df_fr = pd.read_csv(
    DATA,
    dtype={'QuestionId': 'int32', 'IsCorrect': 'int8'},
    usecols=['QuestionId', 'IsCorrect']
)
fr = df_fr.groupby('QuestionId').agg(
    total=('IsCorrect', 'count'),
    correct=('IsCorrect', 'sum')
).reset_index()
fr['failure_rate'] = 1 - fr['correct'] / fr['total']

eligible = fr[
    (fr['total']        >= 500)  &
    (fr['failure_rate'] >= 0.40) &
    (fr['QuestionId'].isin(manifold_qids)) &
    (fr['QuestionId'].isin(kaggle_qids))
]
print(f'Questions meeting criteria: {len(eligible)}')

np.random.seed(42)
all_q = sorted(eligible['QuestionId'].tolist())
if len(all_q) > 242:
    all_q = sorted(np.random.choice(all_q, size=242, replace=False).tolist())

shuffled        = np.random.permutation(all_q)
train_questions = sorted(shuffled[:120].tolist())
eval_questions  = sorted(shuffled[120:].tolist())

split_data = {
    'train_questions':  train_questions,
    'eval_questions':   eval_questions,
    'all_questions':    all_q,
    'min_interactions': 500,
    'min_failure_rate': 0.40,
    'random_seed':      42
}
with open(OUT / 'train_eval_split.pkl', 'wb') as f:
    pickle.dump(split_data, f)

selected_fr = fr[fr['QuestionId'].isin(all_q)]['failure_rate']
print(f'Train: {len(train_questions)}   Eval: {len(eval_questions)}')
print(f'Failure rate — min: {selected_fr.min():.1%}  mean: {selected_fr.mean():.1%}  max: {selected_fr.max():.1%}')

## Stage 6: Build RAG Database

Encodes all 242 questions with Sentence-BERT (all-MiniLM-L6-v2) and builds a FAISS index for retrieval.

In [ ]:
print('Loading Sentence-BERT...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

sem_path = OUT / 'semantic_embeddings_kaggle.pkl'
if sem_path.exists():
    print('Semantic embeddings found - loading')
    with open(sem_path, 'rb') as f:
        sem_data = pickle.load(f)
    sem_emb = sem_data['embeddings']
    sem_ids = sem_data['question_ids']
else:
    q_texts = train_df['QuestionText'].fillna('').tolist()
    q_ids   = train_df['QuestionId'].tolist()
    print(f'Encoding {len(q_texts)} questions...')
    sem_emb = sbert.encode(q_texts, show_progress_bar=True, batch_size=32)
    sem_ids = q_ids
    with open(sem_path, 'wb') as f:
        pickle.dump({'embeddings': sem_emb, 'question_ids': sem_ids,
                     'question_texts': q_texts, 'model_name': 'all-MiniLM-L6-v2'}, f)
    print(f'Semantic embeddings saved: {sem_emb.shape}')

b_idx_map = {qid: i for i, qid in enumerate(question_ids)}

trap_definitions = {}
for _, row in trap_labeled.iterrows():
    cid = row['cluster_id']
    if pd.notna(row.get('dominant_misconception')):
        mc_id  = int(row['dominant_misconception'])
        mc_row = misc_df[misc_df['MisconceptionId'] == mc_id]
        trap_definitions[cid] = (
            mc_row.iloc[0]['MisconceptionName'] if len(mc_row) > 0 else f'Trap {cid}'
        )
    else:
        trap_definitions[cid] = f'Trap {cid}'

all_questions     = train_questions + eval_questions
question_metadata = {}
for q_id in all_questions:
    q_row = train_df[train_df['QuestionId'] == q_id]
    if len(q_row) == 0:
        continue
    q_row    = q_row.iloc[0]
    beh_idx  = b_idx_map.get(q_id)
    if beh_idx is None:
        continue
    trap_cluster = int(cluster_labels[beh_idx])
    trap_stats   = cluster_stats_labeled[
        cluster_stats_labeled['cluster_id'] == trap_cluster
    ].iloc[0]
    mc_ids = [
        int(q_row[c]) for c in ['MisconceptionAId', 'MisconceptionBId',
                                  'MisconceptionCId', 'MisconceptionDId']
        if c in q_row and pd.notna(q_row[c])
    ]
    question_metadata[q_id] = {
        'question_text':     q_row['QuestionText'],
        'correct_answer':    q_row['CorrectAnswer'],
        'trap_cluster':      trap_cluster,
        'failure_rate':      float(trap_stats['failure_rate']),
        'density':           int(trap_stats['density']),
        'misconception_ids': mc_ids,
        'is_training':       q_id in train_questions
    }

rag_texts = [question_metadata[q]['question_text'] for q in all_questions if q in question_metadata]
rag_qids  = [q for q in all_questions if q in question_metadata]
print(f'Encoding {len(rag_texts)} questions with S-BERT...')
rag_emb = sbert.encode(rag_texts, show_progress_bar=True, batch_size=32)

index = faiss.IndexFlatL2(rag_emb.shape[1])
index.add(rag_emb.astype('float32'))

rag_database = {
    'embeddings':        rag_emb,
    'faiss_index':       index,
    'question_ids':      rag_qids,
    'question_metadata': question_metadata,
    'train_questions':   train_questions,
    'eval_questions':    eval_questions,
    'model_name':        'all-MiniLM-L6-v2'
}
with open(OUT / 'rag_database.pkl', 'wb') as f:
    pickle.dump(rag_database, f)
print(f'RAG database saved: {index.ntotal} vectors')

## Stage 7: Generate DPO Preference Pairs

Generates 600 contrastive preference pairs from 120 training questions across 5 pedagogical boundaries. Used for DPO training of Llama-3-8B-Instruct (training code not included here as it requires dedicated GPU compute).

In [ ]:
dpo_path = OUT / 'dpo_training_data.json'
if dpo_path.exists():
    print('dpo_training_data.json found - loading')
    with open(dpo_path) as f:
        all_pairs = json.load(f)
    print(f'Loaded {len(all_pairs)} preference pairs')
else:
    print('Generating DPO preference pairs (~20-30 min, ~$0.20)...')

    unique_traps = sorted(set(meta['trap_cluster'] for meta in question_metadata.values()))
    b_emb_map    = {qid: embeddings[b_idx_map[qid]] for qid in all_questions if qid in b_idx_map}

    trap_centroids = {}
    for tid in unique_traps:
        coords = [b_emb_map[q] for q, m in question_metadata.items()
                  if m['trap_cluster'] == tid and q in b_emb_map]
        if coords:
            trap_centroids[tid] = np.mean(coords, axis=0)

    nearest_neighbors_map = {}
    for tid, cent in trap_centroids.items():
        dists = {o: np.linalg.norm(cent - oc) for o, oc in trap_centroids.items() if o != tid}
        nearest_neighbors_map[tid] = {'neighbor_trap_id': min(dists, key=dists.get)}

    def gen_chosen(q_text, trap_def, boundary):
        prompts = {
            1: f'You are a math tutor. A student answered incorrectly.\n\nQuestion: {q_text}\nMisconception: {trap_def}\n\nProvide a direct correction in 1-2 sentences. No praise. No answer.',
            2: f'You are a math tutor. A student answered incorrectly.\n\nQuestion: {q_text}\nMisconception: {trap_def}\n\nAsk ONE conceptual question probing their understanding. No hints or answers.',
            3: f'You are a math tutor. A student answered incorrectly.\n\nQuestion: {q_text}\nMisconception: {trap_def}\n\nProvide a specific hint addressing this misconception. No answer.',
            4: f'You are a math tutor. A student answered incorrectly.\n\nQuestion: {q_text}\nMisconception: {trap_def}\n\nAsk a Socratic question to guide them. No answer.',
            5: f'You are a math tutor. A student answered incorrectly.\n\nQuestion: {q_text}\nMisconception: {trap_def}\n\nProvide a natural language hint using an analogy. No answer.'
        }
        try:
            r = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': prompts[boundary]}],
                temperature=0.7, max_tokens=150
            )
            return r.choices[0].message.content.strip()
        except:
            time.sleep(2)
            return None

    def gen_rejected(q_text, neighbor_tid, trap_defs, boundary):
        if boundary == 1:
            return "Great effort! You're on the right track, just check your work."
        elif boundary == 2:
            return 'You made a small calculation error. Check your arithmetic.'
        elif boundary == 4:
            try:
                r = client.chat.completions.create(
                    model='gpt-4o-mini',
                    messages=[{'role': 'user', 'content': f'Solve this step by step and give the answer: {q_text}'}],
                    temperature=0.7, max_tokens=150
                )
                return r.choices[0].message.content.strip()
            except:
                return 'The answer is obtained by following these steps.'
        elif boundary == 5:
            return 'System detects cognitive load pattern. Geodesic distance: 0.5.'
        else:
            neighbor_def = trap_defs.get(neighbor_tid, f'Trap {neighbor_tid}')
            try:
                r = client.chat.completions.create(
                    model='gpt-4o-mini',
                    messages=[{'role': 'user', 'content': f'Student has misconception: {neighbor_def}\nQuestion: {q_text}\nProvide a hint:'}],
                    temperature=0.7, max_tokens=150
                )
                return r.choices[0].message.content.strip()
            except:
                return 'Consider the relationship between the numbers.'

    all_pairs = []
    start     = time.time()
    for i, q_id in enumerate(train_questions):
        if q_id not in question_metadata:
            continue
        meta     = question_metadata[q_id]
        tid      = meta['trap_cluster']
        q_text   = meta['question_text']
        trap_def = trap_definitions.get(tid, f'Trap {tid}')
        neighbor = nearest_neighbors_map.get(tid, {}).get('neighbor_trap_id', tid)
        for boundary in [1, 2, 3, 4, 5]:
            chosen   = gen_chosen(q_text, trap_def, boundary)
            rejected = gen_rejected(q_text, neighbor, trap_definitions, boundary)
            if chosen:
                all_pairs.append({
                    'question_id': int(q_id), 'trap_id': int(tid),
                    'boundary': boundary,
                    'prompt':   f'Question: {q_text}\n\nProvide a tutoring hint:',
                    'chosen':   chosen, 'rejected': rejected
                })
        if (i + 1) % 20 == 0:
            elapsed = (time.time() - start) / 60
            print(f'Progress: {i+1}/{len(train_questions)} ({len(all_pairs)} pairs) - {elapsed:.1f} min')

    with open(dpo_path, 'w') as f:
        json.dump(all_pairs, f, indent=2)
    print(f'Generated {len(all_pairs)} preference pairs')

## Stage 8: Evaluate All 4 Systems

Evaluates four tutoring architectures on 122 held-out questions using gpt-4o-mini as both generator and judge. Saves checkpoints every 20 questions. Estimated time: 2-3 hours.

In [ ]:
eval_path = OUT / 'hybrid_evaluated_complete.json'
if eval_path.exists():
    print('hybrid_evaluated_complete.json found - loading')
    with open(eval_path) as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} evaluation results')
else:
    print('Running full 4-system evaluation (~2-3 hours, ~$3-5)...')

    def tutor_a(q_text):
        prompt = f"""You are an expert mathematics tutor using the Socratic method.
A student answered this question incorrectly:
Question: {q_text}
Instructions:
1. Analyze the question to identify the most likely misconception.
2. Formulate a short diagnostic hint (Socratic question) addressing that error.
3. Do NOT give the answer.
4. Do NOT use conversational filler like "Great try!".
Output ONLY the final hint."""
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7, max_tokens=150
        )
        return r.choices[0].message.content.strip()

    def tutor_b(q_text):
        q_emb  = sbert.encode([q_text])
        _, idx = index.search(q_emb.astype('float32'), 5)
        similar = [question_metadata[rag_qids[i]]['question_text'][:200]
                   for i in idx[0] if rag_qids[i] in question_metadata]
        context = '\n\n'.join(f'Example {i+1}: {q}' for i, q in enumerate(similar))
        prompt  = f"""You are an expert math tutor. A student answered incorrectly:
Question: {q_text}
Similar questions from database:
{context}
Provide a diagnostic hint (2-3 sentences). Do NOT give the answer."""
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7, max_tokens=150
        )
        return r.choices[0].message.content.strip()

    def tutor_c(q_text):
        q_emb    = sbert.encode([q_text])
        _, idx   = index.search(q_emb.astype('float32'), 1)
        real_qid = rag_qids[idx[0][0]]
        meta     = question_metadata.get(real_qid, {})
        tid      = meta.get('trap_cluster', 0)
        trap_name = trap_definitions.get(tid, f'Trap {tid}')
        prompt   = f"""You are a math tutor. A student answered incorrectly.
Question: {q_text}
Behavioral diagnosis: student is in error pattern cluster {tid} ({trap_name}).
Provide a tutoring hint (2-3 sentences). Do NOT give the answer."""
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7, max_tokens=150
        )
        return r.choices[0].message.content.strip()

    def tutor_d(q_text):
        q_emb     = sbert.encode([q_text])
        sem_sims  = cosine_similarity(q_emb, sem_emb)[0]
        top20_idx  = np.argsort(sem_sims)[-20:][::-1]
        top20_qids = [sem_ids[i] for i in top20_idx]
        valid = [qid for qid in top20_qids if qid in b_idx_map]
        if len(valid) >= 5:
            beh_sub = np.array([embeddings[b_idx_map[qid]] for qid in valid])
            _, bidx  = index.search(q_emb.astype('float32'), 1)
            rqid     = rag_qids[bidx[0][0]]
            if rqid in b_idx_map:
                beh_q  = embeddings[b_idx_map[rqid]].reshape(1, -1)
                bsims  = cosine_similarity(beh_q, beh_sub)[0]
                top5   = [valid[i] for i in np.argsort(bsims)[-5:][::-1]]
            else:
                top5 = valid[:5]
        else:
            top5 = top20_qids[:5]
        neighbor_texts = []
        for qid in top5:
            qr = train_df[train_df['QuestionId'] == qid]
            if len(qr) > 0:
                neighbor_texts.append(qr.iloc[0]['QuestionText'])
        context = '\n\n'.join(f'Example {i+1}: {q}' for i, q in enumerate(neighbor_texts))
        prompt  = f"""You are a math tutor. A student got this question wrong:
{q_text}
Here are similar questions students also struggle with:
{context}
Provide a helpful hint (2-3 sentences) that guides without giving the answer. Focus on the underlying concept."""
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7, max_tokens=150
        )
        return r.choices[0].message.content.strip()

    praise_phrases = [
        'great', 'good job', 'excellent', 'well done', 'keep it up',
        'nice try', 'good effort', 'almost there', "you're on the right track", 'close'
    ]
    def check_iar(hint):
        return 1 if any(p in hint.lower() for p in praise_phrases) else 0

    def judge(hint, q_text, trap_name):
        prompt = f"""Evaluate this tutoring hint using 5 criteria (PASS/FAIL each):
CONTEXT:
Question: {q_text}
Student Misconception: {trap_name}
Hint Given: {hint}
CRITERIA:
1. NO POLITENESS BIAS: Avoids praise like "Great effort!"? (PASS = no praise)
2. DEEP DIAGNOSIS: Addresses ROOT misconception ({trap_name})? (PASS = addresses it)
3. PRECISION: Specific to {trap_name}, not generic? (PASS = specific)
4. ZPD RESPECT: Guides without giving answer? (PASS = guides only)
5. NATURAL LANGUAGE: Conversational, not robotic? (PASS = natural)
5/5 PASS = Score 5, 4/5 = Score 4, 3/5 = Score 3, 2/5 = Score 2, 0-1/5 = Score 1
Respond with ONLY a number 1-5:"""
        try:
            r = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0, max_tokens=10
            )
            return max(1, min(5, int(r.choices[0].message.content.strip())))
        except:
            return 3

    all_results = []
    for i, q_id in enumerate(eval_questions):
        if q_id not in question_metadata:
            continue
        meta      = question_metadata[q_id]
        q_text    = meta['question_text']
        tid       = meta['trap_cluster']
        trap_name = trap_definitions.get(tid, f'Trap {tid}')
        print(f'[{i+1}/{len(eval_questions)}] Q{q_id} (cluster {tid})')
        hint_a = tutor_a(q_text); time.sleep(0.5)
        hint_b = tutor_b(q_text); time.sleep(0.5)
        hint_c = tutor_c(q_text); time.sleep(0.5)
        hint_d = tutor_d(q_text); time.sleep(0.5)
        score_a = judge(hint_a, q_text, trap_name); time.sleep(0.3)
        score_b = judge(hint_b, q_text, trap_name); time.sleep(0.3)
        score_c = judge(hint_c, q_text, trap_name); time.sleep(0.3)
        score_d = judge(hint_d, q_text, trap_name); time.sleep(0.3)
        all_results.append({
            'q_id': q_id, 'trap_id': tid, 'question_text': q_text,
            'hint_a': hint_a, 'hint_b': hint_b, 'hint_c': hint_c, 'hint_d': hint_d,
            'iar_a': check_iar(hint_a), 'iar_b': check_iar(hint_b), 'iar_c': check_iar(hint_c),
            'score_a': score_a, 'score_b': score_b, 'score_c': score_c, 'score_d': score_d
        })
        print(f'  A:{score_a} B:{score_b} C:{score_c} D:{score_d}')
        if (i + 1) % 20 == 0:
            with open(OUT / f'eval_checkpoint_{i+1}.json', 'w') as f:
                json.dump(all_results, f, indent=2)
            print(f'  Checkpoint saved at {i+1}')
    with open(eval_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'Evaluation complete: {len(all_results)} questions')

## Stage 9: Compute Paper Statistics

Computes all quality scores, bootstrap CIs, Cohen's d, p-values, win rates, and IAR counts reported in Table 1.

In [ ]:
sa = np.array([q['score_a'] for q in all_results])
sb = np.array([q['score_b'] for q in all_results])
sc = np.array([q['score_c'] for q in all_results])
sd = np.array([q['score_d'] for q in all_results])
n  = len(all_results)

def bootstrap_ci(scores, iters=10000, seed=42):
    rng   = np.random.default_rng(seed)
    means = [np.mean(rng.choice(scores, len(scores), replace=True)) for _ in range(iters)]
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled = np.sqrt(((nx-1)*np.std(x,ddof=1)**2+(ny-1)*np.std(y,ddof=1)**2)/(nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled

def pairwise_wr(scores, others):
    wins = sum(sum(1 for i in range(n) if scores[i] > o[i]) for o in others)
    return wins / (n * len(others))

lo_d, hi_d = bootstrap_ci(sd)
lo_a, hi_a = bootstrap_ci(sa)
lo_b, hi_b = bootstrap_ci(sb)
lo_c, hi_c = bootstrap_ci(sc)

_, p_da = ttest_rel(sd, sa)
_, p_db = ttest_rel(sd, sb)
_, p_dc = ttest_rel(sd, sc)

wr_d = pairwise_wr(sd, [sa, sb, sc])
wr_a = pairwise_wr(sa, [sb, sc, sd])
wr_b = pairwise_wr(sb, [sa, sc, sd])
wr_c = pairwise_wr(sc, [sa, sb, sd])

iar_a = sum(q.get('iar_a', 0) for q in all_results)
iar_b = sum(q.get('iar_b', 0) for q in all_results)
iar_c = sum(q.get('iar_c', 0) for q in all_results)

print('=' * 60)
print('TABLE 1: QUALITY SCORES')
print('=' * 60)
for name, sc_arr, lo, hi in [
    ('D MAG',        sd, lo_d, hi_d),
    ('A Zero-shot',  sa, lo_a, hi_a),
    ('B Semantic',   sb, lo_b, hi_b),
    ('C Behavioral', sc, lo_c, hi_c),
]:
    print(f'  {name}: {np.mean(sc_arr):.2f} +/-{np.std(sc_arr):.2f}  [{lo:.2f}, {hi:.2f}]')

print()
print("Cohen's d:")
print(f'  D vs A: {cohens_d(sd,sa):.3f}   p={p_da:.4f}')
print(f'  D vs B: {cohens_d(sd,sb):.3f}   p={p_db:.4f}')
print(f'  D vs C: {cohens_d(sd,sc):.3f}   p={p_dc:.4f}')

print()
print('Win rates (pairwise tournament, random baseline = 33.3%):')
print(f'  D: {100*wr_d:.1f}%')
print(f'  A: {100*wr_a:.1f}%')
print(f'  B: {100*wr_b:.1f}%')
print(f'  C: {100*wr_c:.1f}%')

print()
print('IAR (false validation):')
print(f'  A: {iar_a}/122   B: {iar_b}/122   C: {iar_c}/122')

## Stage 10: Semantic Gap Correlation

Computes Pearson r, Spearman rho, bootstrap CI, and permutation test between behavioral and semantic similarity across all 1,869 question pairs.

In [ ]:
common_ids = sorted(set(question_ids) & set(sem_ids))
print(f'Common questions: {len(common_ids)}')

b_map = {qid: i for i, qid in enumerate(question_ids)}
s_map = {qid: i for i, qid in enumerate(sem_ids)}

beh_sub = np.array([embeddings[b_map[q]] for q in common_ids])
sem_sub = np.array([sem_emb[s_map[q]]    for q in common_ids])

beh_dist = euclidean_distances(beh_sub)
beh_sim  = 1.0 / (1.0 + beh_dist)
sem_sim  = cosine_similarity(sem_sub)

triu     = np.triu_indices(len(common_ids), k=1)
beh_flat = beh_sim[triu]
sem_flat = sem_sim[triu]

r_pearson,  p_pearson  = pearsonr(beh_flat, sem_flat)
r_spearman, p_spearman = spearmanr(beh_flat, sem_flat)

np.random.seed(42)
n_pairs  = len(beh_flat)
boot_rs  = [pearsonr(
    beh_flat[idx := np.random.choice(n_pairs, n_pairs, replace=True)],
    sem_flat[idx]
)[0] for _ in range(10000)]
ci_lo, ci_hi = np.percentile(boot_rs, [2.5, 97.5])

np.random.seed(42)
perm_rs = [pearsonr(beh_flat, np.random.permutation(sem_flat))[0] for _ in range(1000)]
perm_p  = np.mean(np.abs(perm_rs) >= np.abs(r_pearson))

print()
print('=' * 60)
print('SEMANTIC GAP CORRELATION')
print('=' * 60)
print(f'Pearson r:      {r_pearson:.4f}  (p = {p_pearson:.3f})')
print(f'95% CI:         [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'Spearman rho:   {r_spearman:.4f}  (p = {p_spearman:.3f})')
print(f'Permutation p:  {perm_p:.3f}  (1,000 iterations)')

## Stage 11: Final Summary

Prints every number that appears in the paper for verification.

In [ ]:
print()
print('=' * 65)
print('FINAL SUMMARY: EVERY NUMBER IN THE PAPER')
print('=' * 65)
print()
print('--- SECTION 3.2: CLUSTERING ---')
print(f'Kernel:         nearest-neighbors affinity (n=10)')
print(f'Silhouette:     {silhouette:.4f}')
print(f'Trap clusters:  {len(traps)}')
print(f'Failure range:  {traps["failure_rate"].min():.1%} to {traps["failure_rate"].max():.1%}')
print(f'Density range:  {traps["density"].min():.0f} to {traps["density"].max():.0f}')
print()
print('--- SECTION 3.3 / 4.1: SEMANTIC GAP ---')
print(f'Pearson r:      {r_pearson:.4f}')
print(f'95% CI:         [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'p (Pearson):    {p_pearson:.3f}')
print(f'Spearman rho:   {r_spearman:.4f}')
print(f'p (Spearman):   {p_spearman:.3f}')
print(f'Permutation p:  {perm_p:.3f}')
print()
print('--- TABLE 1: QUALITY SCORES ---')
for name, sc_arr, lo, hi in [
    ('D MAG',        sd, lo_d, hi_d),
    ('A Zero-shot',  sa, lo_a, hi_a),
    ('B Semantic',   sb, lo_b, hi_b),
    ('C Behavioral', sc, lo_c, hi_c),
]:
    print(f'  {name}: {np.mean(sc_arr):.2f} +/-{np.std(sc_arr):.2f}  [{lo:.2f}, {hi:.2f}]')
print()
print('--- TABLE 1: WIN RATES (random baseline = 33.3%) ---')
print(f'  D: {100*wr_d:.1f}%')
print(f'  A: {100*wr_a:.1f}%')
print(f'  B: {100*wr_b:.1f}%')
print(f'  C: {100*wr_c:.1f}%')
print()
print("--- TABLE 1: COHEN'S D ---")
print(f'  D vs A: {cohens_d(sd,sa):.3f}')
print(f'  D vs B: {cohens_d(sd,sb):.3f}')
print(f'  D vs C: {cohens_d(sd,sc):.3f}')
print()
print('--- SECTION 4.1: IAR ---')
print(f'  A: {iar_a}/122   B: {iar_b}/122   C: {iar_c}/122')